# Inspect extracted player statistics

Each league uses its own latest saved date. Zero is filled data; NULL and blank
strings are missing. Players normally have data only for positions they played.
Historical snapshots can have empty grouped columns. `P` and `DH` source splits
are included in inspection when stored, although defender reports exclude them.

In [1]:
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd

from mlb_airflow_data_pipeline.db_utils import read_player_stats
from mlb_airflow_data_pipeline.statsapi_parameters_script import (
    DATA_FILE_LOCATION,
    FIELDING_POSITIONS,
    PLAYER_INFORMATION,
    expected_output_columns,
)

pd.set_option("display.max_rows", None)

# Edit this path to inspect a different database.
database_path = Path(DATA_FILE_LOCATION) / "mlb_data.db"

## Load the schema and latest rows

This cell reads one consistent view of the database. Rerun it to refresh the data
following another extraction. No database file is created if the path is wrong.

In [2]:
with closing(
    sqlite3.connect(database_path.resolve().as_uri() + "?mode=ro", uri=True)
) as connection:
    schema = pd.read_sql_query('PRAGMA table_info("player_stats")', connection)
    latest_snapshots = pd.read_sql_query(
        "SELECT league_name, MAX(date) AS date FROM player_stats GROUP BY league_name",
        connection,
    )
    latest_players = pd.concat(
        [
            read_player_stats(connection, league_name, execution_date)
            for league_name, execution_date in latest_snapshots.itertuples(
                index=False, name=None
            )
        ],
        ignore_index=True,
    ).sort_values(["league_name", "team_id", "player_id"])

snapshot_counts = (
    latest_players.groupby(["league_name", "date"])
    .size()
    .rename("total_rows")
    .reset_index()
)
snapshot_counts

,league_name,date,total_rows
0,american_league,2026-10-11,535
1,national_league,2026-10-10,556


## Check the new schema

The first table counts grouped columns. The next lists their names and SQLite
types and whether they are supported by the extraction schema; edit `schema_prefix` to inspect hitting, pitching, or a particular position.
Columns are added when extraction encounters them, so not every possible source
statistic must exist in the database.

In [10]:
grouped_schema = schema.loc[
    schema["name"].str.startswith(("hitting_", "pitching_", "fielding_")),
    ["name", "type"],
].sort_values("name")
grouped_schema["supported"] = grouped_schema["name"].isin(expected_output_columns())
grouped_schema[~grouped_schema["supported"]]

,name,type,supported


In [4]:
schema_counts = (
    grouped_schema["name"]
    .str.split("_", n=1)
    .str[0]
    .value_counts()
    .reindex(["hitting", "pitching", "fielding"], fill_value=0)
    .rename_axis("group")
    .rename("columns")
    .reset_index()
)
schema_counts

,group,columns
0,hitting,34
1,pitching,62
2,fielding,159


## Check filled fielding columns

Coverage shows the number of populated rows for **every stored fielding column**,
using each league's latest snapshot. The following position summary counts players
with any populated fielding statistic at that position. Missing positions remain
visible with zero rows; sparse values alone do not indicate an extraction failure.

In [5]:
fielding_columns = grouped_schema.loc[
    grouped_schema["name"].str.startswith("fielding_"), "name"
].tolist()
fielding_values = latest_players[fielding_columns].astype("string")
filled = fielding_values.notna() & fielding_values.apply(
    lambda column: column.str.strip().ne("")
)
snapshot_keys = [latest_players["league_name"], latest_players["date"]]
fielding_counts = filled.groupby(snapshot_keys).sum()
coverage = (
    fielding_counts.rename_axis(columns="column")
    .stack()
    .rename("filled_rows")
    .reset_index()
    .merge(snapshot_counts, on=["league_name", "date"])
)
coverage

,league_name,date,column,filled_rows,total_rows
0,american_league,2026-10-11,fielding_1B_age,65,535
1,american_league,2026-10-11,fielding_1B_assists,65,535
2,american_league,2026-10-11,fielding_1B_chances,65,535
3,american_league,2026-10-11,fielding_1B_doublePlays,65,535
4,american_league,2026-10-11,fielding_1B_errors,65,535
5,american_league,2026-10-11,fielding_1B_fielding,65,535
6,american_league,2026-10-11,fielding_1B_games,65,535
7,american_league,2026-10-11,fielding_1B_gamesPlayed,65,535
8,american_league,2026-10-11,fielding_1B_gamesStarted,65,535
9,american_league,2026-10-11,fielding_1B_innings,65,535


In [6]:
position_flags = pd.DataFrame(index=latest_players.index)
for position in FIELDING_POSITIONS:
    columns = [
        column
        for column in fielding_columns
        if column.startswith(f"fielding_{position}_")
    ]
    position_flags[position] = filled[columns].any(axis=1)
position_coverage = position_flags.groupby(snapshot_keys).sum()
position_coverage

,,P,C,1B,2B,3B,SS,LF,CF,RF,DH
league_name,date,,,,,,,,,,
american_league,2026-10-11,294,43,65,75,76,61,103,72,88,184
national_league,2026-10-10,0,0,0,0,0,0,0,0,0,0


## Inspect individual players

Edit `league_name`, `position`, and `sample_limit`, then rerun this cell. All stored
statistics for the chosen position are included, so missing values can be checked
alongside player identity. `latest_players`, `coverage`, and `schema` remain
available for additional queries in new cells.

In [7]:
league_name = "american_league"
position = "C"
sample_limit = 10

position_columns = [
    column for column in fielding_columns if column.startswith(f"fielding_{position}_")
]
sample_rows = latest_players["league_name"].eq(league_name) & filled[
    position_columns
].any(axis=1)
player_samples = latest_players.loc[
    sample_rows,
    ["league_name", "date", "player_id"] + PLAYER_INFORMATION + position_columns,
].head(sample_limit)
player_samples

,league_name,date,player_id,playername,team_id,fielding_C_age,fielding_C_assists,fielding_C_catcherERA,fielding_C_catchersInterference,fielding_C_caughtStealing,...,fielding_C_passedBall,fielding_C_pickoffs,fielding_C_putOuts,fielding_C_rangeFactorPer9Inn,fielding_C_rangeFactorPerGame,fielding_C_stolenBasePercentage,fielding_C_stolenBases,fielding_C_throwingErrors,fielding_C_triplePlays,fielding_C_wildPitches
0,american_league,2026-10-11,518595,Vaughn Grissom,108,37,11,4.17,0,6,...,0,0,221,9.90,8.29,.727,16,0,0,9
3,american_league,2026-10-11,623168,Moisés Ballesteros,108,35,27,4.09,2,19,...,4,0,540,9.85,7.77,.689,42,2,0,14
7,american_league,2026-10-11,665861,Zach Neto,108,27,6,3.74,2,1,...,0,0,145,9.44,6.29,.938,15,1,0,5
18,american_league,2026-10-11,682515,Yusei Kikuchi,108,30,1,9.00,0,0,...,0,0,7,4.50,4.00,1.000,2,0,0,1
27,american_league,2026-10-11,694208,Trevor Martin,108,22,9,4.39,0,4,...,3,0,182,8.86,6.59,.826,19,2,0,8
43,american_league,2026-10-11,650968,Yohel Pozo,110,29,4,9.53,2,4,...,0,0,94,9.09,3.92,.692,9,0,0,4
46,american_league,2026-10-11,665966,Jackson Holliday,110,27,36,3.75,2,11,...,4,0,731,9.46,8.25,.814,48,5,0,15
70,american_league,2026-10-11,694212,Yennier Cano,110,21,25,3.69,0,12,...,3,2,495,9.05,8.25,.739,34,1,0,16
85,american_league,2026-10-11,657136,Alec Gamboa,111,30,26,3.47,0,8,...,3,0,544,9.00,7.81,.855,47,4,0,17
90,american_league,2026-10-11,668939,Adley Rutschman,111,28,36,3.17,2,21,...,3,2,626,8.83,8.17,.661,41,5,0,20
